### Data Cleaning v01

In [101]:
import pandas as pd

In [102]:
df = pd.read_csv('data/variant_summary.txt', sep='\t', low_memory=False)
df.head()

,#AlleleID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,LastEvaluated,RS# (dbSNP),...,AlternateAlleleVCF,SomaticClinicalImpact,SomaticClinicalImpactLastEvaluated,ReviewStatusClinicalImpact,Oncogenicity,OncogenicityLastEvaluated,ReviewStatusOncogenicity,SCVsForAggregateGermlineClassification,SCVsForAggregateSomaticClinicalImpact,SCVsForAggregateOncogenicityClassification
0,15041,Indel,NM_014855.3(AP5Z1):c.80_83delinsTGCTGTAAACTGTA...,9907,AP5Z1,HGNC:22197,Pathogenic/Likely pathogenic,1,"Dec 17, 2024",397704705,...,TGCTGTAAACTGTAACTGTAAA,-,-,-,-,-,-,SCV001451119|SCV005622007|SCV005909190,-,-
1,15041,Indel,NM_014855.3(AP5Z1):c.80_83delinsTGCTGTAAACTGTA...,9907,AP5Z1,HGNC:22197,Pathogenic/Likely pathogenic,1,"Dec 17, 2024",397704705,...,TGCTGTAAACTGTAACTGTAAA,-,-,-,-,-,-,SCV001451119|SCV005622007|SCV005909190,-,-
2,15042,Deletion,NM_014855.3(AP5Z1):c.1413_1426del (p.Leu473fs),9907,AP5Z1,HGNC:22197,Pathogenic,1,"Jun 29, 2010",397704709,...,G,-,-,-,-,-,-,SCV000020156,-,-
3,15042,Deletion,NM_014855.3(AP5Z1):c.1413_1426del (p.Leu473fs),9907,AP5Z1,HGNC:22197,Pathogenic,1,"Jun 29, 2010",397704709,...,G,-,-,-,-,-,-,SCV000020156,-,-
4,15043,single nucleotide variant,NM_014630.3(ZNF592):c.3136G>A (p.Gly1046Arg),9640,ZNF592,HGNC:28986,Uncertain significance,0,"Jun 29, 2015",150829393,...,A,-,-,-,-,-,-,SCV000020157,-,-


In [103]:
print(df.shape)
print(df['ClinicalSignificance'].value_counts())

(8671400, 43)
ClinicalSignificance
Uncertain significance                  4553933
Likely benign                           2046721
-                                        491670
Benign                                   423294
Pathogenic                               391266
                                         ...   
Likely benign; association                    2
Likely pathogenic; protective                 2
Benign; Affects; association; other           2
confers sensitivity; other                    2
Established risk allele; association          2
Name: count, Length: 96, dtype: int64


In [104]:
# there are many ambiguous labels based on the ClinicalSignificance value counts
# resolution is to exclude those from the used data
labels_to_include = ['Pathogenic', 'Likely pathogenic', 'Benign', 'Likely benign']
df = df[df['ClinicalSignificance'].isin(labels_to_include)]

In [105]:
# focusing on simple nucleotide variant for now
df = df[df['Type'] == 'single nucleotide variant']

In [106]:
print(df['ReviewStatus'].value_counts())

# filter for high-quality review status only
high_quality = [
    'practice guideline',
    'reviewed by expert panel',
    'criteria provided, multiple submitters, no conflicts'
]
df = df[df['ReviewStatus'].isin(high_quality)]

ReviewStatus
criteria provided, single submitter                     2040459
criteria provided, multiple submitters, no conflicts     466434
no assertion criteria provided                           104245
reviewed by expert panel                                  22255
practice guideline                                           38
Name: count, dtype: int64


In [107]:
# check for missing data in columns of interest
important_cols = ['GeneSymbol', 'ClinicalSignificance', 'Assembly', 'ReferenceAlleleVCF', 'AlternateAlleleVCF']
print(df[important_cols].isnull().sum())

GeneSymbol              0
ClinicalSignificance    0
Assembly                0
ReferenceAlleleVCF      0
AlternateAlleleVCF      0
dtype: int64


In [108]:
# check genome builds
print(df['Assembly'].value_counts())

# pick one assembly to avoid duplicates
# GRCh38 is the current reference
df = df[df['Assembly'] == 'GRCh38']

Assembly
GRCh38    244363
GRCh37    244362
na             2
Name: count, dtype: int64


In [109]:
# check whether #AlleleID deduplication actually removes anything
n_before = len(df)
n_after = df['#AlleleID'].nunique()
print(f'Rows before dedup: {n_before:,}')
print(f'Unique AlleleIDs:   {n_after:,}')
print(f'Rows removed:       {n_before - n_after:,}')

# check VariationID as alternative key
print(f'\nUnique VariationIDs: {df["VariationID"].nunique():,}')

# if rows are removed, inspect them
dupes = df[df.duplicated(subset=['#AlleleID'], keep=False)]
if len(dupes) > 0:
    print(f'\nDuplicate AlleleID rows ({len(dupes):,}):')
    print(dupes[['#AlleleID', 'VariationID', 'ClinicalSignificance']].sort_values('#AlleleID').head(10))
else:
    print('\nNo duplicate AlleleIDs found — dedup step has no effect.')

# drop duplicate variants
df = df.drop_duplicates(subset=['#AlleleID'], keep='first')

Rows before dedup: 244,363
Unique AlleleIDs:   244,250
Rows removed:       113

Unique VariationIDs: 244,250

Duplicate AlleleID rows (226):
        #AlleleID  VariationID ClinicalSignificance
18026       24911         9872           Pathogenic
18027       24911         9872           Pathogenic
18047       24917         9878           Pathogenic
18048       24917         9878           Pathogenic
103387      98999        93092               Benign
103388      98999        93092               Benign
194236     178249       178714               Benign
194237     178249       178714               Benign
194244     178251       178716               Benign
194245     178251       178716               Benign


In [110]:
print(df['Chromosome'].value_counts())

# remove non-standard instances for the sake of simplicity
# keep 1-22, X, and Y
standard = [str(i) for i in range(1, 23)] + ['X', 'Y']
df = df[df['Chromosome'].isin(standard)]

# drop rows where allele is missing (ClinVar uses 'na' as a placeholder)
df = df[(df['ReferenceAlleleVCF'] != 'na') & (df['AlternateAlleleVCF'] != 'na')]

Chromosome
2     22620
1     20499
17    17088
19    13951
11    13749
16    13499
3     13110
12    12371
7     12027
9     11472
5     10281
10     9329
15     9128
X      9118
14     8227
4      7858
8      7360
13     7038
6      6583
22     5648
20     5515
18     4218
21     3352
MT      206
Y         3
Name: count, dtype: int64


In [111]:
df = df.reset_index(drop=True)

In [112]:
df.head()

,#AlleleID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,LastEvaluated,RS# (dbSNP),...,AlternateAlleleVCF,SomaticClinicalImpact,SomaticClinicalImpactLastEvaluated,ReviewStatusClinicalImpact,Oncogenicity,OncogenicityLastEvaluated,ReviewStatusOncogenicity,SCVsForAggregateGermlineClassification,SCVsForAggregateSomaticClinicalImpact,SCVsForAggregateOncogenicityClassification
0,15044,single nucleotide variant,NM_017547.4(FOXRED1):c.694C>T (p.Gln232Ter),55572,FOXRED1,HGNC:26927,Pathogenic,1,"Aug 17, 2025",267606829,...,T,-,-,-,-,-,-,SCV000680696|SCV001363290|SCV002793147|SCV0029...,-,-
1,15066,single nucleotide variant,NM_001042472.3(ABHD12):c.1054C>T (p.Arg352Ter),26090,ABHD12,HGNC:15868,Pathogenic,1,"Jan 03, 2024",267606624,...,A,-,-,-,-,-,-,SCV001379909|SCV002061204|SCV004152554|SCV0048...,-,-
2,15069,single nucleotide variant,NM_138413.4(HOGA1):c.860G>T (p.Gly287Val),112817,HOGA1,HGNC:25155,Pathogenic,1,"Oct 03, 2025",138207257,...,T,-,-,-,-,-,-,SCV000915488|SCV000937843|SCV002555640|SCV0038...,-,-
3,15074,single nucleotide variant,NM_001201543.2(FAM161A):c.685C>T (p.Arg229Ter),84140,FAM161A,HGNC:25808,Pathogenic,1,"Oct 28, 2024",267606794,...,A,-,-,-,-,-,-,SCV001239599|SCV001246773|SCV001586471|SCV0019...,-,-
4,15075,single nucleotide variant,NM_001201543.2(FAM161A):c.1309A>T (p.Arg437Ter),84140,FAM161A,HGNC:25808,Pathogenic,1,"Mar 07, 2025",200691042,...,A,-,-,-,-,-,-,SCV000229237|SCV000329583|SCV000894286|SCV0009...,-,-


In [113]:
# filter out relevant features
features_to_keep = [
    'GeneSymbol',
    'Chromosome',
    'Start',
    'Stop',
    'ReferenceAlleleVCF',
    'AlternateAlleleVCF',
    'NumberSubmitters',
]

# separate into target and feature tensors
label_map = {
    'Pathogenic': 1, 'Likely pathogenic': 1,
    'Benign': 0, 'Likely benign': 0
}
y = df['ClinicalSignificance'].map(label_map).rename('label')
X = df[features_to_keep].copy()
print('Features:')
print(X.head())
print('\nMissing values:')
print(X.isnull().sum())
print('\nTarget:')
print(y.head())
print('\nClass distribution:')
print(y.value_counts())

Features:
  GeneSymbol Chromosome      Start       Stop ReferenceAlleleVCF  \
0    FOXRED1         11  126275389  126275389                  C   
1     ABHD12         20   25302322   25302322                  G   
2      HOGA1         10   97611535   97611535                  G   
3    FAM161A          2   61840319   61840319                  G   
4    FAM161A          2   61839695   61839695                  T   

  AlternateAlleleVCF  NumberSubmitters  
0                  T                 6  
1                  A                 5  
2                  T                12  
3                  A                11  
4                  A                23  

Missing values:
GeneSymbol            0
Chromosome            0
Start                 0
Stop                  0
ReferenceAlleleVCF    0
AlternateAlleleVCF    0
NumberSubmitters      0
dtype: int64

Target:
0    1
1    1
2    1
3    1
4    1
Name: label, dtype: int64

Class distribution:
label
0    211773
1     32270
Name: count, dty

In [114]:
# convert to JSON and save
y.to_frame().to_json('./data/target.json', orient='records')
X.to_json('./data/features.json', orient='records')